In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# 1. Device Configuration (Ensure GPU is used as requested)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 2. Hyperparameters
BATCH_SIZE = 64
LEARNING_RATE = 0.001
EPOCHS = 5

# 3. Load MNIST Dataset
# We use a transform to convert images to PyTorch Tensors
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) # Standard MNIST Normalization
])

# Download and load training data
train_dataset = torchvision.datasets.MNIST(root='./data', train=True,
                                           transform=transform, download=True)
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Download and load test data
test_dataset = torchvision.datasets.MNIST(root='./data', train=False,
                                          transform=transform, download=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 4. Define CNN Architecture
# (Conv -> ReLU -> Pool -> Conv -> ReLU -> Pool -> FC)
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        # Layer 1: Conv (1 in, 32 out) -> ReLU -> MaxPool
        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        # Layer 2: Conv (32 in, 64 out) -> ReLU -> MaxPool
        self.layer2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=0),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        # Fully Connected Layer
        self.fc = nn.Linear(64 * 6 * 6, 10) # 6x6 is the spatial dimension after pooling

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.size(0), -1) # Flatten
        out = self.fc(out)
        return out

model = CNN().to(device)
print(model)

Using device: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 38.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.09MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.30MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.44MB/s]


CNN(
  (layer1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Linear(in_features=2304, out_features=10, bias=True)
)


In [2]:
# 1. Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 2. Training Loop
print(f"Starting training on {device}...")
total_step = len(train_loader)
loss_list = []
acc_list = []

for epoch in range(EPOCHS):
    model.train()
    for i, (images, labels) in enumerate(train_loader):
        # Move tensors to the configured device (GPU)
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i + 1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{EPOCHS}], Step [{i+1}/{total_step}], Loss: {loss.item():.4f}')
            loss_list.append(loss.item())

# 3. Evaluation (Test Accuracy)
model.eval()  # Set to evaluation mode (batchnorm/dropout behavior changes)
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'\nTest Accuracy of the CNN on the 10000 test images: {accuracy:.2f}%')

# 4. Save Model (Optional, good practice)
torch.save(model.state_dict(), 'cnn_mnist.ckpt')

Starting training on cuda...
Epoch [1/5], Step [100/938], Loss: 0.2528
Epoch [1/5], Step [200/938], Loss: 0.1037
Epoch [1/5], Step [300/938], Loss: 0.0231
Epoch [1/5], Step [400/938], Loss: 0.0474
Epoch [1/5], Step [500/938], Loss: 0.2066
Epoch [1/5], Step [600/938], Loss: 0.1069
Epoch [1/5], Step [700/938], Loss: 0.1687
Epoch [1/5], Step [800/938], Loss: 0.0121
Epoch [1/5], Step [900/938], Loss: 0.0060
Epoch [2/5], Step [100/938], Loss: 0.0311
Epoch [2/5], Step [200/938], Loss: 0.0191
Epoch [2/5], Step [300/938], Loss: 0.0033
Epoch [2/5], Step [400/938], Loss: 0.0067
Epoch [2/5], Step [500/938], Loss: 0.2176
Epoch [2/5], Step [600/938], Loss: 0.0449
Epoch [2/5], Step [700/938], Loss: 0.0493
Epoch [2/5], Step [800/938], Loss: 0.0037
Epoch [2/5], Step [900/938], Loss: 0.0870
Epoch [3/5], Step [100/938], Loss: 0.0012
Epoch [3/5], Step [200/938], Loss: 0.0250
Epoch [3/5], Step [300/938], Loss: 0.0490
Epoch [3/5], Step [400/938], Loss: 0.0160
Epoch [3/5], Step [500/938], Loss: 0.0189
Epoch

In [6]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset

# 1. Custom Dataset Adapter
class MNISTDetection(Dataset):
    def __init__(self, root, train=True, transform=None):
        self.mnist = torchvision.datasets.MNIST(root=root, train=train, download=True)
        self.transform = transform

    def __len__(self):
        return len(self.mnist)

    def __getitem__(self, idx):
        img, label = self.mnist[idx]
        
        # Convert grayscale to RGB (3 channels) so it fits the pre-trained backbone
        img = img.convert("RGB")
        
        if self.transform:
            img = self.transform(img)

        # Define the Bounding Box [x_min, y_min, x_max, y_max]
        # Since the digit is the whole image, we box the whole 28x28 area
        boxes = torch.tensor([[0, 0, 28, 28]], dtype=torch.float32)
        
        # Labels: Class 0 is usually background, so we shift digits to 1-10
        # digit 0 -> label 1, digit 1 -> label 2, etc.
        labels = torch.tensor([label + 1], dtype=torch.int64)
        
        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        target["image_id"] = torch.tensor([idx])
        target["area"] = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        target["iscrowd"] = torch.zeros((1,), dtype=torch.int64)

        return img, target

# 2. Prepare Data Loaders
# Collate function is needed because Faster R-CNN expects a list of images/targets
def collate_fn(batch):
    return tuple(zip(*batch))

transform = transforms.ToTensor()

train_ds = MNISTDetection(root='./data', train=True, transform=transform)
test_ds = MNISTDetection(root='./data', train=False, transform=transform)

# Use a smaller batch size (Faster R-CNN is heavy on VRAM)
train_loader_rcnn = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)
test_loader_rcnn = DataLoader(test_ds, batch_size=8, shuffle=False, collate_fn=collate_fn)

# 3. Load and Modify Pre-trained Model
# We load a model pre-trained on COCO
model_rcnn = fasterrcnn_resnet50_fpn(weights='DEFAULT')

# Replace the predictor head
# Num classes = 10 digits + 1 background = 11
num_classes = 11
in_features = model_rcnn.roi_heads.box_predictor.cls_score.in_features
model_rcnn.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

model_rcnn.to(device)

# 4. Training Loop
# Faster R-CNN returns a loss dictionary automatically during training
params = [p for p in model_rcnn.parameters() if p.requires_grad]
optimizer_rcnn = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

print("Starting Faster R-CNN Training (this is slower than CNN)...")
model_rcnn.train()

# Train for fewer epochs because it's slow
for epoch in range(1): # Increase epochs if you have time/GPU power
    total_loss = 0
    for i, (images, targets) in enumerate(train_loader_rcnn):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model_rcnn(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer_rcnn.zero_grad()
        losses.backward()
        optimizer_rcnn.step()

        total_loss += losses.item()

        if (i+1) % 100 == 0:
            print(f"Epoch [{epoch+1}], Step [{i+1}], Loss: {losses.item():.4f}")

print("Training Complete!")

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100%|██████████| 160M/160M [00:00<00:00, 232MB/s] 


Starting Faster R-CNN Training (this is slower than CNN)...
Epoch [1], Step [100], Loss: 0.0841


KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader

# 1. Prepare Data with Resizing (Required for VGG/AlexNet)
# VGG expects 224x224 RGB images
transform_transfer = transforms.Compose([
    transforms.Resize((224, 224)),      # Resize 28 -> 224
    transforms.Grayscale(num_output_channels=3), # 1 channel -> 3 channels
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Use a smaller batch size because 224x224 images take a lot of VRAM
train_dataset = MNIST(root='./data', train=True, transform=transform_transfer, download=True)
test_dataset = MNIST(root='./data', train=False, transform=transform_transfer, download=True)

train_loader_tf = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader_tf = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 2. Load Pre-trained VGG16
print("Loading VGG16...")
# Weights='DEFAULT' loads the best available pre-trained weights
model_vgg = models.vgg16(weights='DEFAULT')

# 3. Freeze Early Layers (Optional but recommended for speed)
# We prevent the early feature extraction layers from updating
for param in model_vgg.features.parameters():
    param.requires_grad = False

# 4. Modify the Classifier
# VGG16's classifier ends with a Linear layer (4096 -> 1000)
# We replace it with (4096 -> 10)
num_features = model_vgg.classifier[6].in_features
model_vgg.classifier[6] = nn.Linear(num_features, 10)

model_vgg = model_vgg.to(device)

# 5. Training Loop
criterion = nn.CrossEntropyLoss()
# Only optimize the parameters that require gradients (the classifier)
optimizer_vgg = optim.Adam(model_vgg.parameters(), lr=0.001)

print("Starting VGG16 Fine-Tuning...")
epochs = 1  # 1 Epoch is usually enough for high accuracy on MNIST with Transfer Learning

for epoch in range(epochs):
    model_vgg.train()
    for i, (images, labels) in enumerate(train_loader_tf):
        images, labels = images.to(device), labels.to(device)

        optimizer_vgg.zero_grad()
        outputs = model_vgg(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_vgg.step()

        if (i+1) % 50 == 0:
            print(f"Step [{i+1}/{len(train_loader_tf)}], Loss: {loss.item():.4f}")

# 6. Evaluation
model_vgg.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader_tf:
        images, labels = images.to(device), labels.to(device)
        outputs = model_vgg(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"VGG16 Test Accuracy: {100 * correct / total:.2f}%")

In [ ]:
partie 2:

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=1, patch_size=7, embed_dim=64, img_size=28):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim=64, num_heads=4):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.fc_out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        batch_size, n_patches, _ = x.shape
        qkv = self.qkv(x).reshape(batch_size, n_patches, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        energy = (q @ k.transpose(-2, -1)) * self.scale
        attn = energy.softmax(dim=-1)
        
        out = (attn @ v).transpose(1, 2).reshape(batch_size, n_patches, self.embed_dim)
        out = self.fc_out(out)
        return out

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim=64, num_heads=4, mlp_dim=128, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class ViT(nn.Module):
    def __init__(self, in_channels=1, patch_size=7, embed_dim=64, num_heads=4, mlp_dim=128, num_layers=4, num_classes=10, img_size=28):
        super().__init__()
        self.patch_embed = PatchEmbedding(in_channels, patch_size, embed_dim, img_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + self.patch_embed.num_patches, embed_dim))
        self.transformer = nn.Sequential(*[TransformerBlock(embed_dim, num_heads, mlp_dim) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(embed_dim)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        batch_size = x.shape[0]
        x = self.patch_embed(x)
        cls_token = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = x + self.pos_embed
        x = self.transformer(x)
        x = self.norm(x)
        return self.fc(x[:, 0])

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

model = ViT().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.003)

for epoch in range(5):
    model.train()
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        if (i+1) % 200 == 0:
            print(f'Epoch [{epoch+1}/5], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'ViT Accuracy: {100 * correct / total:.2f}%')

Epoch [1/5], Step [200/938], Loss: 0.6409
Epoch [1/5], Step [400/938], Loss: 0.3575
Epoch [1/5], Step [600/938], Loss: 0.2184
Epoch [1/5], Step [800/938], Loss: 0.2294
Epoch [2/5], Step [200/938], Loss: 0.1547
Epoch [2/5], Step [400/938], Loss: 0.2183
Epoch [2/5], Step [600/938], Loss: 0.0601
Epoch [2/5], Step [800/938], Loss: 0.2055
Epoch [3/5], Step [200/938], Loss: 0.0795
Epoch [3/5], Step [400/938], Loss: 0.2870
Epoch [3/5], Step [600/938], Loss: 0.3912
Epoch [3/5], Step [800/938], Loss: 0.0950
Epoch [4/5], Step [200/938], Loss: 0.1945
Epoch [4/5], Step [400/938], Loss: 0.0629
Epoch [4/5], Step [600/938], Loss: 0.0746
Epoch [4/5], Step [800/938], Loss: 0.1438
Epoch [5/5], Step [200/938], Loss: 0.0852
Epoch [5/5], Step [400/938], Loss: 0.0987
Epoch [5/5], Step [600/938], Loss: 0.0314
Epoch [5/5], Step [800/938], Loss: 0.1549
ViT Accuracy: 97.45%
